# CLASSIFICATION- ML methods 

## 1. Loading data (and and necessary libraries)

In [3]:
#necessary libraries

import pandas as pd 
import numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import balanced_accuracy_score
import pickle
import statsmodels.formula.api as smf 
from statsmodels.stats.anova import anova_lm
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import RandomOverSampler, SMOTE, ADASYN
from imblearn.combine import SMOTETomek
from imblearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler
from imblearn.under_sampling import TomekLinks
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import cross_validate
from sklearn.metrics import recall_score, precision_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.metrics import accuracy_score, precision_score, recall_score

In [4]:
#loading training and test data
train_df = pd.read_csv("insurance_train.csv")
test_df = pd.read_csv("insurance_test.csv")

In [5]:
train_df['reward_zero'] = train_df['reward'].apply(lambda x: 'yes' if x == 0 else 'no')
test_df['reward_zero'] = test_df['reward'].apply(lambda x: 'yes' if x == 0 else 'no')

In [6]:
train_df = train_df[train_df["customer_score"] != 1.0]


In [7]:
categorical_cols = [ 'person_gender', 'entity_type', 'channel', 'agent_id', 'entity_a', 'location', 'product_id']
numerical_cols = ['reward', 'support_interactions', 'customer_score', 'person_age', 'revenue', 'trip_length']

In [8]:
#age - replace 100 values(probably unknown) by median
median_age = train_df['person_age'].median()

train_df.loc[train_df['person_age'] == 100, 'person_age'] = median_age
test_df.loc[test_df['person_age'] == 100, 'person_age'] = median_age


In [9]:
#trip_length
median_singapore_tripguard = train_df[
    (train_df['location'] == 'singapore') & 
    (train_df['product_id'] == 'TripGuard TicketCover')
]['trip_length'].median()

def replace_trip_lengths_above_4000(df, median_value):
    df.loc[df['trip_length'] > 4000, 'trip_length'] = median_value
    return df

train_df = replace_trip_lengths_above_4000(train_df, median_singapore_tripguard)
test_df = replace_trip_lengths_above_4000(test_df, median_singapore_tripguard)


In [10]:
#location -> others
threshold = int(len(train_df) * 0.005)
location_counts = train_df['location'].value_counts()
locations_to_keep = location_counts[location_counts > threshold].index
train_df['location'] = train_df['location'].apply(lambda x: x if x in locations_to_keep else 'Other')
test_df['location'] = test_df['location'].apply(lambda x: x if x in locations_to_keep else 'Other')


In [11]:
#product_id gruping
def group_product(prod):
    if "TripGuard" in prod:
        return "TripGuard"
    elif "SecurePlan" in prod:
        return "SecurePlan"
    elif "DriveSafe" in prod:
        return "DriveSafe"
    elif "TravelShield" in prod:
        return "TravelShield"
    elif "QuickProtect" in prod:
        return "QuickProtect"
    elif "CruiseSafe" in prod:
        return "CruiseSafe"
    else:
        return "Other"


train_df["prod_id_grouped"] = train_df["product_id"].apply(group_product)
test_df["prod_id_grouped"] = test_df["product_id"].apply(group_product)



In [12]:
train_df = train_df.drop(columns=['product_id'])
test_df = test_df.drop(columns=['product_id'])

In [13]:
#remove chanel 
train_df = train_df.drop(columns=['channel'])
test_df = test_df.drop(columns=['channel'])



In [14]:
#remove support_interactions and person age based on low correlation results 
train_df = train_df.drop(columns=['support_interactions'])
test_df = test_df.drop(columns=['support_interactions'])
train_df = train_df.drop(columns=['person_age'])
test_df = test_df.drop(columns=['person_age'])

In [15]:
X = train_df.drop("claim_status", axis=1)
y = train_df["claim_status"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)



In [16]:
categorical_cols = [ 'person_gender', 'entity_type', 'agent_id', 'entity_a', 'location', 'prod_id_grouped']

In [17]:
#one-hot encoding
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoder.fit(X_train[categorical_cols])

X_train_encoded = encoder.transform(X_train[categorical_cols])
X_val_encoded   = encoder.transform(X_val[categorical_cols])
test_encoded    = encoder.transform(test_df[categorical_cols])


In [18]:
model = joblib.load("estimator.pkl")
preds = model.predict(test_df)
test_df_with_preds = test_df.copy()
test_df_with_preds["prediction"] = preds
test_df_with_preds.to_csv("insurance_test_with_predictions.csv", index=False)

C:\Users\Admin\anaconda3\Lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.6.1 when using version 1.5.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\Admin\anaconda3\Lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.5.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\Admin\anaconda3\Lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator OneHotEncoder from version 1.6.1 when using version 1.5.1. This might lead to breaking c